In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Line Loading Limits

This example demonstrates three ways of restricting how heavily a line may be loaded in a PyPSA optimisation: a static `s_max_pu`, a time-varying `s_max_pu`, and a voltage angle difference limit, first derived by hand as an equivalent `s_max_pu` and then applied natively via `Line.v_ang_max`.

We build a small 3-bus meshed AC network with two generators and one load, which is enough to have interesting (and reroutable) power flows.

In [ ]:
import numpy as np
import pandas as pd

import pypsa

n = pypsa.Network()
n.set_snapshots(range(4))

n.add("Bus", ["bus0", "bus1", "bus2"], v_nom=380)
n.add("Line", "line0", bus0="bus0", bus1="bus1", x=15, r=1.5, s_nom=100)
n.add("Line", "line1", bus0="bus1", bus1="bus2", x=20, r=2.0, s_nom=100)
n.add("Line", "line2", bus0="bus2", bus1="bus0", x=10, r=1.0, s_nom=100)
n.add("Generator", "gen0", bus="bus0", p_nom=200, marginal_cost=10)
n.add("Generator", "gen1", bus="bus1", p_nom=200, marginal_cost=30)
n.add("Load", "load0", bus="bus2", p_set=120)

n.calculate_dependent_values()
n.plot(title="3-bus meshed AC network")

## 1. Static N-1 Approximation via `s_max_pu`

The simplest way to build a security margin into a linear optimal power flow is to derate every line below its thermal limit, e.g. to 70% of `s_nom`, by setting a static `Line.s_max_pu`. This leaves headroom so that if a parallel circuit trips, the remaining lines are less likely to be overloaded.

This is only a cheap **approximation** of true N-1 security: it does not check any specific outage, and a well-loaded network can still violate limits after a real contingency. For the exact method, which explicitly enforces line ratings for every single branch outage, see the [Security-Constrained LOPF example](scigrid-sclopf.ipynb) and the [contingencies user-guide page](../user-guide/optimization/contingencies.md).

In [ ]:
n.lines["s_max_pu"] = 0.7

n.optimize()

loading = n.lines_t.p0.loc[0].abs() / n.lines.s_nom
loading.rename("loading [p.u.]").to_frame()

The most heavily used line (`line2`) is loaded at exactly 70%, confirming that the static derating caps the flow at `0.7 * s_nom` even though the full `s_nom` would otherwise have been used.

## 2. Dynamic Line Rating via Time-Varying `s_max_pu`

Overhead line ratings depend on ambient conditions such as wind speed and air temperature: a well-cooled line can safely carry more current than its static rating suggests, while a hot, still day requires a lower limit. This dynamic line rating (DLR) is modelled in PyPSA by supplying a per-snapshot series in `n.lines_t.s_max_pu` instead of (or in addition to) the static `Line.s_max_pu`.

We give `line2` a rating that varies over the four snapshots, mimicking changing weather conditions.

In [ ]:
n.lines["s_max_pu"] = 1.0
n.lines_t.s_max_pu["line2"] = pd.Series([0.9, 0.8, 0.7, 0.85], index=n.snapshots)

n.optimize()

flow = n.lines_t.p0["line2"].abs()
cap = n.lines_t.s_max_pu["line2"] * n.lines.at["line2", "s_nom"]
pd.DataFrame({"flow": flow, "cap": cap})

In [ ]:
pd.DataFrame({"flow": flow, "cap": cap}).plot.bar(
    ylabel="MW", title="line2: flow vs. time-varying rating"
)

In every snapshot the dispatch is rerouted just enough that `line2`'s flow tracks its time-varying cap, illustrating how a dynamic line rating can be used to unlock (or restrict) capacity as weather conditions change, without touching the line's nominal `s_nom`.

## 3. Voltage Angle Difference Limits

In the linearized power flow used for the LOPF, the voltage angle difference across a line is directly proportional to its power flow:

$$\theta_{bus0} - \theta_{bus1} = x_{pu,eff} \cdot s$$

where `x_pu_eff` is the line's effective per-unit reactance (available on `n.lines.x_pu_eff` after calling `n.calculate_dependent_values()`) and `s` is the flow in MW, since PyPSA's per-unit system uses a base power of 1 MVA. This means a maximum allowed voltage angle difference `v_ang_max` (in degrees) can be translated directly into an equivalent `s_max_pu`:

$$s_{max,pu} = \frac{\mathrm{deg2rad}(v_{ang,max})}{x_{pu,eff} \cdot s_{nom}}$$

To see the mechanism, we first apply this conversion by hand, capping `line2` at a target angle difference of 0.3 degrees.

In [ ]:
n.lines["s_max_pu"] = 1.0
n.lines_t.s_max_pu = n.lines_t.s_max_pu.drop(columns="line2")

v_ang_max = 0.3  # degrees
x_pu_eff = n.lines.at["line2", "x_pu_eff"]
s_nom = n.lines.at["line2", "s_nom"]
s_max_pu = np.deg2rad(v_ang_max) / (x_pu_eff * s_nom)
n.lines.loc["line2", "s_max_pu"] = s_max_pu
s_max_pu

In [ ]:
n.optimize()

n.generators_t.p_set = n.generators_t.p
n.lpf()

v_ang = n.buses_t.v_ang.loc[0]
angle_diff = np.rad2deg(v_ang["bus2"] - v_ang["bus0"])
angle_diff

The resulting angle difference across `line2` matches the target magnitude of 0.3 degrees (the sign follows the flow direction along the line), confirming that the `s_max_pu` conversion enforces the desired voltage angle difference limit.

### Native `v_ang_min` / `v_ang_max`

The same limit can be set directly on the line via `Line.v_ang_min` and `Line.v_ang_max` (in degrees). Internally, the optimisation adds exactly the flow bounds derived above, so the result is identical to the manual conversion.

In [ ]:
n.lines.loc["line2", "s_max_pu"] = 1.0
n.lines.loc["line2", "v_ang_max"] = v_ang_max

n.optimize()

n.generators_t.p_set = n.generators_t.p
n.lpf()

v_ang = n.buses_t.v_ang.loc[0]
np.rad2deg(v_ang["bus2"] - v_ang["bus0"])

!!! info "Native voltage angle difference limits"

    Enforcement of `Line.v_ang_min` and `Line.v_ang_max` in the optimisation is available from the next release (see [GitHub issue #1481](https://github.com/PyPSA/PyPSA/issues/1481)). The manual `s_max_pu` conversion shown above is exactly the mechanism used internally and remains a useful way to understand it; on earlier versions it is the way to apply such a limit. Transformers are not yet covered, since their angle difference couples with the optimisable `phase_shift`.